# Lecture 13: Multiple Regression and Workflow Design

**Case:** Which Vienna hotels appear inexpensive relative to their location and
observable quality?

This extends the simple regression from Lecture 12. We specify candidate
models, compare them, construct honest interpretations, and distinguish a
confidence interval for a mean from a prediction interval for one hotel.

Adapted from Gábor Békés and Gábor Kézdi's advanced hotel-regression materials
and Chapter 10 case study.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import root_mean_squared_error

sns.set_theme(style="whitegrid")

In [ ]:
hotels = pd.read_csv("../../data/raw/hotels_course.csv")
vienna = hotels.query("city == 'Vienna' and accommodation == 'Hotel'").copy()
vienna[["price", "distance_miles", "rating", "stars", "review_count"]].describe().round(2)

## 1. Translate the question into model specifications

Distance is the baseline. Rating and star category are observable quality
measures. We add them in stages so that every change has a substantive reason.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=vienna, x="rating", y="log_price", hue="stars", ax=axes[0])
sns.boxplot(data=vienna, x="stars", y="log_price", ax=axes[1])
axes[0].set(title="Ratings and log price")
axes[1].set(title="Log price by star category")
plt.tight_layout()
plt.show()

In [ ]:
model_1 = smf.ols("log_price ~ distance_miles", data=vienna).fit(cov_type="HC3")
model_2 = smf.ols("log_price ~ distance_miles + rating", data=vienna).fit(cov_type="HC3")
model_3 = smf.ols(
    "log_price ~ distance_miles + rating + C(stars)", data=vienna
).fit(cov_type="HC3")

In [ ]:
def model_row(name, model):
    rmse = root_mean_squared_error(vienna["log_price"], model.fittedvalues)
    return {"model": name, "adj_r2": model.rsquared_adj, "aic": model.aic, "rmse_log": rmse}

comparison = pd.DataFrame(
    [model_row("distance", model_1), model_row("+ rating", model_2), model_row("+ stars", model_3)]
)
comparison.round(3)

Adjusted R-squared and AIC describe in-sample fit with different complexity
penalties. They are useful here, but they are not substitutes for honest
out-of-sample evaluation; that arrives in Lecture 17.

In [ ]:
terms = ["distance_miles", "rating", "C(stars)[T.3.5]", "C(stars)[T.4.0]"]
estimates = pd.DataFrame(
    {"estimate": model_3.params, "robust_se": model_3.bse}
).loc[terms]
estimates.round(3)

In [ ]:
distance_pct = 100 * (np.exp(model_3.params["distance_miles"]) - 1)
rating_pct = 100 * (np.exp(model_3.params["rating"]) - 1)
print(f"One additional mile is associated with {distance_pct:.1f}% different price.")
print(f"One rating point is associated with {rating_pct:.1f}% different price.")

## 2. Predict one hotel and express uncertainty

In [ ]:
proposed_hotel = pd.DataFrame(
    {"distance_miles": [2.5], "rating": [4.2], "stars": [4.0]}
)
prediction_log = model_3.get_prediction(proposed_hotel).summary_frame(alpha=0.05)
prediction_log.round(3)

In [ ]:
smearing_factor = np.mean(np.exp(model_3.resid))
columns = ["mean", "mean_ci_lower", "mean_ci_upper", "obs_ci_lower", "obs_ci_upper"]
prediction_price = np.exp(prediction_log[columns]) * smearing_factor
prediction_price.round(0)

The confidence interval describes uncertainty about the expected price for
hotels with these features. The prediction interval is wider because an
individual hotel's price also contains irreducible variation.

## 3. Use residuals to find candidates—not conclusions

In [ ]:
vienna["predicted_log_price"] = model_3.fittedvalues
vienna["residual_log_price"] = model_3.resid
vienna["predicted_price"] = np.exp(model_3.fittedvalues) * smearing_factor

In [ ]:
candidates = vienna.nsmallest(8, "residual_log_price")[
    ["hotel_id", "price", "predicted_price", "distance_miles", "stars", "rating"]
]
candidates.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=vienna, x="predicted_price", y="price", alpha=0.65, ax=ax)
limits = [vienna["price"].min(), vienna["price"].max()]
ax.plot(limits, limits, color="black", linestyle="--")
ax.set(title="Observed versus predicted prices", xlabel="Predicted price", ylabel="Observed price")
plt.show()

## Session 1 reconstruction handoff

Only after completing both regression notebooks, open
`prompts/session01-regression-reconstruction.md`. The class will specify the
analysis in plain language, ask an LLM to reconstruct it, and audit the result
against this human-built reference.

### Optional extension: external validity

Fit the same formula separately in Amsterdam and Barcelona. Compare the
distance coefficient and prediction error. A stable formula is not guaranteed
to have stable coefficients in another market.

In [ ]:
city_models = {
    city: smf.ols("log_price ~ distance_miles + rating + C(stars)", data=group).fit()
    for city, group in hotels.query("accommodation == 'Hotel'").groupby("city")
}
pd.DataFrame(
    {city: {"distance": model.params["distance_miles"], "adj_r2": model.rsquared_adj}
     for city, model in city_models.items()}
).T.round(3)